# 03: Spatial cross-validation check

This notebook compares random stratified cross-validation with spatial group cross-validation. It is a robustness diagnostic for spatial autocorrelation and spatial transferability. If spatial CV accuracy is substantially lower than random CV, the manuscript should discuss possible inflation of random validation accuracy.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'config' / 'paper1_config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from coffeemap.config import load_config, ensure_project_dirs
from coffeemap.io import find_file, read_table, write_table
from coffeemap.schema import find_first_column
from coffeemap.spatial_cv import find_coordinate_columns, infer_feature_columns, compare_random_vs_spatial_cv

CONFIG = load_config(ROOT / 'config' / 'paper1_config.yaml')
PATHS = ensure_project_dirs(CONFIG, ROOT)
RANDOM_SEED = int(CONFIG['project'].get('random_seed', 42))
print('Project root:', ROOT)


Project root: d:\2024_PhD_Research\Chap2_Mapping_Coffee\Code_workflow


## 1. Load coordinate-enabled sample table

Preferred input: `Table_TrainSamples_RF_Final_2024.csv` with class labels, x/y or lon/lat coordinates, and numeric predictor columns. If the file is missing, the notebook writes a status table rather than silently producing invalid results.

In [ ]:
input_dir = PATHS['input_dir']
supp_dir = PATHS['supplementary_dir']
fig_dir = PATHS['figures_dir']

sample_candidates = [
    'Table_TrainSamples_RF_Final_2024.csv',
    'train_samples.csv',
    'training_samples.csv',
    'Table_AllSamples_RF_Final_2024.csv',
    'all_samples.csv',
]
sample_path = find_file(sample_candidates, [input_dir], required=False)

if sample_path is None:
    status = pd.DataFrame([{
        'status': 'not_available',
        'reason': 'No training/all-sample table with coordinates was found in data/raw/.',
        'required_action': 'Export a table with class label, coordinates, and predictor values.'
    }])
    write_table(status, supp_dir / 'Table_SpatialCV_Status.csv')
    (supp_dir / 'spatial_cv_note.txt').write_text(
        'Spatial cross-validation was not run because no coordinate-enabled training sample table was available.\n',
        encoding='utf-8'
    )
    print(status.to_string(index=False))
    df_samples = None
else:
    df_samples = read_table(sample_path)
    print('Loaded:', sample_path)
    print('Shape:', df_samples.shape)
    display(df_samples.head())


## 2. Audit labels, coordinates and predictors

In [3]:
if df_samples is not None:
    label_col = find_first_column(df_samples, CONFIG['columns']['true_label_candidates'])
    x_col, y_col = find_coordinate_columns(df_samples)
    if label_col is None:
        raise ValueError('Could not identify a class label column.')
    if x_col is None or y_col is None:
        status = pd.DataFrame([{
            'status': 'not_available',
            'reason': 'Sample table found, but coordinate columns were not detected.',
            'detected_label_col': label_col,
            'required_action': 'Add x/y, lon/lat, or UTM coordinate columns to the sample export.'
        }])
        write_table(status, supp_dir / 'Table_SpatialCV_Status.csv')
        can_run_spatial_cv = False
    else:
        feature_cols = infer_feature_columns(df_samples, label_col=label_col, coord_cols=[x_col, y_col])
        audit = pd.DataFrame([{
            'sample_file': sample_path.name,
            'n_rows': len(df_samples),
            'label_col': label_col,
            'x_col': x_col,
            'y_col': y_col,
            'n_candidate_features': len(feature_cols),
            'n_classes': df_samples[label_col].nunique()
        }])
        write_table(audit, supp_dir / 'Table_SpatialCV_InputAudit.csv')
        print(audit.to_string(index=False))
        print('First 20 candidate features:', feature_cols[:20])
        can_run_spatial_cv = len(feature_cols) >= 2 and df_samples[label_col].nunique() >= 2
else:
    can_run_spatial_cv = False


## 3. Compare random versus spatial CV

In [4]:
if can_run_spatial_cv:
    result = compare_random_vs_spatial_cv(
        df_samples,
        label_col=label_col,
        feature_cols=feature_cols,
        x_col=x_col,
        y_col=y_col,
        n_splits=5,
        random_state=RANDOM_SEED,
        n_estimators=500,
    )
    if result.status != 'ok':
        status = pd.DataFrame([{'status': result.status, 'reason': result.message}])
        write_table(status, supp_dir / 'Table_SpatialCV_Status.csv')
        print(result.message)
    else:
        write_table(result.summary, supp_dir / 'Table_SpatialCV_Summary.csv')
        write_table(result.fold_scores, supp_dir / 'Table_SpatialCV_FoldScores.csv')
        print(result.message)
        display(result.summary)
else:
    result = None
    print('Spatial CV not run. See Table_SpatialCV_Status.csv if generated.')


Spatial CV not run. See Table_SpatialCV_Status.csv if generated.
